In [1]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path
from datetime import datetime

In [2]:
BASE_DIR = Path("E:/Product_Analyst")

RAW_DIR = BASE_DIR / "Data/raw"
PROCESSED_DIR = BASE_DIR / "Data/processed"
LOG_DIR = BASE_DIR / "automation/logs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
events = pd.read_csv(RAW_DIR / "product_app_events_raw.csv")
orders = pd.read_csv(RAW_DIR / "product_orders_raw.csv")

print("Raw datasets loaded")
print("Events:", events.shape)
print("Orders:", orders.shape)

Raw datasets loaded
Events: (12035, 12)
Orders: (3612, 16)


In [5]:
from pathlib import Path

automation_path = Path("E:/Product_Analyst/automation")

print("Folder exists:", automation_path.exists())
print("Files inside folder:")

if automation_path.exists():
    for file in automation_path.iterdir():
        print(file.name)

Folder exists: True
Files inside folder:
logs


In [6]:
import sys
import importlib

automation_path = "E:/Product_Analyst/automation"

if automation_path not in sys.path:
    sys.path.insert(0, automation_path)

importlib.invalidate_caches()

print("Path added:", automation_path)
print("Python can find module:", 
      importlib.util.find_spec("data_pipeline") is not None)

Path added: E:/Product_Analyst/automation
Python can find module: False


In [7]:
from pathlib import Path

automation_path = Path("E:/Product_Analyst/automation")

for file in automation_path.iterdir():
    print("Name:", repr(file.name))
    print("Suffix:", repr(file.suffix))
    print("Is file:", file.is_file())

Name: 'logs'
Suffix: ''
Is file: False


In [8]:
from pathlib import Path

automation_path = Path("E:/Product_Analyst/automation")

pipeline_file = automation_path / "data_pipeline.py"

pipeline_file.touch()

print("Created:", pipeline_file)
print("Exists:", pipeline_file.exists())

Created: E:\Product_Analyst\automation\data_pipeline.py
Exists: True


In [9]:
from pathlib import Path

pipeline_file = Path("E:/Product_Analyst/automation/data_pipeline.py")

print("File exists:", pipeline_file.exists())
print("File size:", pipeline_file.stat().st_size, "bytes")

File exists: True
File size: 0 bytes


In [11]:

import pandas as pd
import numpy as np

# Product catalogue
product_mapping = {
    "Wireless Earbuds": "P1001",
    "Smart Watch": "P1002",
    "Bluetooth Speaker": "P1003",
    "Running Shoes": "P2001",
    "Cotton T-Shirt": "P2002",
    "Backpack": "P2003",
    "Mixer Grinder": "P3001",
    "Water Bottle": "P3002",
    "Bedsheet Set": "P3003",
    "Face Serum": "P4001",
    "Sunscreen SPF 50": "P4002",
    "Yoga Mat": "P5001",
    "Cricket Bat": "P5002",
    "Data Analytics Handbook": "P6001",
    "Python Programming": "P6002"
}

def clean_events(df):
    df = df.drop_duplicates().copy()

    df["event_timestamp"] = pd.to_datetime(
        df["event_timestamp"], errors="coerce"
    )

    df["event_name"] = (
        df["event_name"].astype("string").str.strip().str.lower()
    )

    df["city"] = df["city"].astype("string").str.strip()
    df["product_name"] = df["product_name"].astype("string").str.strip()

    missing_id = df["product_id"].isna()

    df.loc[missing_id, "product_id"] = (
        df.loc[missing_id, "product_name"].map(product_mapping)
    )

    return df


def clean_orders(df):
    df = df.drop_duplicates().copy()

    df["order_date"] = pd.to_datetime(
        df["order_date"], errors="coerce"
    )

    df["city"] = df["city"].astype("string").str.strip()
    df["product_name"] = df["product_name"].astype("string").str.strip()

    gross = pd.to_numeric(df["gross_amount_inr"], errors="coerce")
    net = pd.to_numeric(df["net_amount_inr"], errors="coerce")

    calculated_discount = ((gross - net) / gross * 100).where(
        gross > 0
    )

    df["discount_pct_clean"] = (
        pd.to_numeric(df["discount_pct"], errors="coerce")
        .fillna(calculated_discount)
        .round(2)
    )

    return df



In [13]:
from pathlib import Path

pipeline_file = Path("E:/Product_Analyst/automation/data_pipeline.py")

print("File size:", pipeline_file.stat().st_size)

print("\nFile contents:")
print(pipeline_file.read_text(encoding="utf-8"))

File size: 0

File contents:



In [14]:
import sys
import importlib

sys.path.insert(0, "E:/Product_Analyst/automation")

import data_pipeline

importlib.reload(data_pipeline)

print([
    name for name in dir(data_pipeline)
    if name.startswith("clean_")
])

[]


In [15]:
from pathlib import Path

file_path = Path("E:/Product_Analyst/automation/data_pipeline.py")

print("File size:", file_path.stat().st_size, "bytes")
print("File contents:")
print(repr(file_path.read_text(encoding="utf-8")))

File size: 0 bytes
File contents:
''


In [16]:

from pathlib import Path

pipeline_code = '''
import pandas as pd
import numpy as np

product_mapping = {
    "Wireless Earbuds": "P1001",
    "Smart Watch": "P1002",
    "Bluetooth Speaker": "P1003",
    "Running Shoes": "P2001",
    "Cotton T-Shirt": "P2002",
    "Backpack": "P2003",
    "Mixer Grinder": "P3001",
    "Water Bottle": "P3002",
    "Bedsheet Set": "P3003",
    "Face Serum": "P4001",
    "Sunscreen SPF 50": "P4002",
    "Yoga Mat": "P5001",
    "Cricket Bat": "P5002",
    "Data Analytics Handbook": "P6001",
    "Python Programming": "P6002"
}

def clean_events(df):
    df = df.drop_duplicates().copy()

    df["event_timestamp"] = pd.to_datetime(
        df["event_timestamp"], errors="coerce"
    )

    df["event_name"] = (
        df["event_name"].astype("string").str.strip().str.lower()
    )

    df["city"] = df["city"].astype("string").str.strip()
    df["product_name"] = df["product_name"].astype("string").str.strip()

    missing_id = df["product_id"].isna()

    df.loc[missing_id, "product_id"] = (
        df.loc[missing_id, "product_name"].map(product_mapping)
    )

    return df


def clean_orders(df):
    df = df.drop_duplicates().copy()

    df["order_date"] = pd.to_datetime(
        df["order_date"], errors="coerce"
    )

    df["city"] = df["city"].astype("string").str.strip()
    df["product_name"] = df["product_name"].astype("string").str.strip()

    gross = pd.to_numeric(df["gross_amount_inr"], errors="coerce")
    net = pd.to_numeric(df["net_amount_inr"], errors="coerce")

    calculated_discount = ((gross - net) / gross * 100).where(
        gross > 0
    )

    df["discount_pct_clean"] = (
        pd.to_numeric(df["discount_pct"], errors="coerce")
        .fillna(calculated_discount)
        .round(2)
    )

    return df
'''

file_path = Path("E:/Product_Analyst/automation/data_pipeline.py")
file_path.write_text(pipeline_code, encoding="utf-8")

print("Python functions saved successfully!")
print("File size:", file_path.stat().st_size, "bytes")


Python functions saved successfully!
File size: 1812 bytes


In [17]:
import sys
import importlib

sys.path.insert(0, "E:/Product_Analyst/automation")

import data_pipeline
importlib.reload(data_pipeline)

from data_pipeline import clean_events, clean_orders

print("Both functions imported successfully!")

Both functions imported successfully!


In [18]:
events_clean = clean_events(events)
orders_clean = clean_orders(orders)

print("Data cleaning completed!")
print("Events:", events_clean.shape)
print("Orders:", orders_clean.shape)

Data cleaning completed!
Events: (12000, 12)
Orders: (3600, 17)


In [19]:
events_clean.to_csv(
    PROCESSED_DIR / "product_app_events_clean.csv",
    index=False
)

orders_clean.to_csv(
    PROCESSED_DIR / "product_orders_clean.csv",
    index=False
)

print("Cleaned datasets exported successfully!")

Cleaned datasets exported successfully!


In [20]:
delivered = orders_clean[
    orders_clean["order_status"].str.strip().str.lower() == "delivered"
].copy()

kpi_report = {
    "total_orders": len(orders_clean),
    "delivered_orders": len(delivered),
    "unique_customers": delivered["user_id"].nunique(),
    "units_sold": delivered["quantity"].sum(),
    "delivered_revenue_inr": round(
        delivered["net_amount_inr"].sum(), 2
    ),
    "average_order_value_inr": round(
        delivered["net_amount_inr"].mean(), 2
    )
}

kpi_df = pd.DataFrame(
    list(kpi_report.items()),
    columns=["KPI", "Value"]
)

display(kpi_df)

,KPI,Value
0,total_orders,3600.00
1,delivered_orders,2550.00
2,unique_customers,1639.00
3,units_sold,3298.00
4,delivered_revenue_inr,4280663.88
5,average_order_value_inr,1678.69


In [21]:
kpi_df.to_csv(
    PROCESSED_DIR / "automated_kpi_report.csv",
    index=False
)

print("KPI report generated successfully!")

KPI report generated successfully!


In [22]:
import logging
from datetime import datetime

LOG_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=LOG_DIR / "notebook_pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

logging.info("Product Analytics pipeline started")
logging.info(f"Events processed: {len(events_clean)}")
logging.info(f"Orders processed: {len(orders_clean)}")
logging.info(f"Delivered revenue: {kpi_report['delivered_revenue_inr']}")

print("Pipeline execution details logged!")

Pipeline execution details logged!


In [23]:
log_file = LOG_DIR / "notebook_pipeline.log"

print(log_file.read_text(encoding="utf-8"))

2026-10-30 22:35:25,216 - INFO - Product Analytics pipeline started
2026-10-30 22:35:25,217 - INFO - Events processed: 12000
2026-10-30 22:35:25,217 - INFO - Orders processed: 3600
2026-10-30 22:35:25,217 - INFO - Delivered revenue: 4280663.88



In [24]:
quality_checks = {
    "Duplicate event IDs": events_clean["event_id"].duplicated().sum(),
    "Duplicate order IDs": orders_clean["order_id"].duplicated().sum(),
    "Missing event timestamps": events_clean["event_timestamp"].isna().sum(),
    "Missing order dates": orders_clean["order_date"].isna().sum(),
    "Negative order quantities": (orders_clean["quantity"] < 0).sum(),
    "Negative net amounts": (orders_clean["net_amount_inr"] < 0).sum(),
    "Missing clean discounts": orders_clean["discount_pct_clean"].isna().sum()
}

quality_df = pd.DataFrame(
    list(quality_checks.items()),
    columns=["Quality Check", "Issue Count"]
)

display(quality_df)

,Quality Check,Issue Count
0,Duplicate event IDs,0
1,Duplicate order IDs,0
2,Missing event timestamps,0
3,Missing order dates,0
4,Negative order quantities,0
5,Negative net amounts,0
6,Missing clean discounts,0


In [25]:
quality_df.to_csv(
    PROCESSED_DIR / "data_quality_report.csv",
    index=False
)

logging.info("Data quality report generated")

print("Data quality report saved successfully!")

Data quality report saved successfully!


In [26]:
def run_product_analytics_pipeline():

    try:
        logging.info("Pipeline execution started")

        # Load raw datasets
        events = pd.read_csv(RAW_DIR / "product_app_events_raw.csv")
        orders = pd.read_csv(RAW_DIR / "product_orders_raw.csv")

        # Clean datasets
        events_clean = clean_events(events)
        orders_clean = clean_orders(orders)

        # Validate
        assert events_clean["event_id"].is_unique
        assert orders_clean["order_id"].is_unique

        assert events_clean["event_timestamp"].notna().all()
        assert orders_clean["order_date"].notna().all()

        # Export cleaned datasets
        events_clean.to_csv(
            PROCESSED_DIR / "product_app_events_clean.csv",
            index=False
        )

        orders_clean.to_csv(
            PROCESSED_DIR / "product_orders_clean.csv",
            index=False
        )

        # Generate KPI report
        delivered = orders_clean[
            orders_clean["order_status"].str.strip().str.lower()
            == "delivered"
        ]

        kpi_report = {
            "total_orders": len(orders_clean),
            "delivered_orders": len(delivered),
            "unique_customers": delivered["user_id"].nunique(),
            "units_sold": delivered["quantity"].sum(),
            "delivered_revenue_inr": round(
                delivered["net_amount_inr"].sum(), 2
            ),
            "average_order_value_inr": round(
                delivered["net_amount_inr"].mean(), 2
            )
        }

        kpi_df = pd.DataFrame(
            list(kpi_report.items()),
            columns=["KPI", "Value"]
        )

        kpi_df.to_csv(
            PROCESSED_DIR / "automated_kpi_report.csv",
            index=False
        )

        logging.info("Pipeline completed successfully")

        print("Pipeline completed successfully!")
        display(kpi_df)

    except Exception:
        logging.exception("Pipeline execution failed")
        print("Pipeline failed. Check the log file.")
        raise

In [27]:
run_product_analytics_pipeline()

Pipeline completed successfully!


,KPI,Value
0,total_orders,3600.00
1,delivered_orders,2550.00
2,unique_customers,1639.00
3,units_sold,3298.00
4,delivered_revenue_inr,4280663.88
5,average_order_value_inr,1678.69


In [28]:
%pip install papermill

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import papermill as pm

print("Papermill installed and ready!")

Papermill installed and ready!


In [33]:
from pathlib import Path

base_dir = Path("E:/Product_Analyst")

for file in base_dir.rglob("*.ipynb"):
    print(file)

In [32]:
reports_dir = Path("E:/Product_Analyst/reports")

reports_dir.mkdir(parents=True, exist_ok=True)

print("Reports folder ready:", reports_dir.exists())

Reports folder ready: True


In [34]:
from pathlib import Path

base_dir = Path("E:/Product_Analyst")

for file in base_dir.rglob("*.ipynb"):
    print(file)

In [35]:
from pathlib import Path

print("Current notebook working directory:")
print(Path.cwd())

print("\nFiles in current directory:")
for file in Path.cwd().iterdir():
    print(file.name)

Current notebook working directory:
C:\Users\kollu

Files in current directory:
.angular-config.json
.arduinoIDE
.bash_history
.cache
.conda
.condarc
.continuum
.copilot
.dbclient
.eclipse
.emulator_console_auth_token
.ghcp-appmod
.ghcp-appmod-java
.gitconfig
.gradle
.ipynb_checkpoints
.ipython
.jupyter
.lemminx
.m2
.mamba
.matplotlib
.node_repl_history
.p2
.sts4
.templateengine
.virtual_documents
.vscode
.vscode-shared
anaconda3
AppData
Application Data
BANK CUSTOMER CHURN ANALYSIS.ipynb
battery-report.html
Contacts
Cookies
Data Cleaning.ipynb
Data Understanding.ipynb
data_cleaning.ipynb
data_validation.ipynb
deansdashboard
demo
Documents
Downloads
eclipse
eclipse-workspace
edb_pem_server.exe
edb_pgagent_pg18.exe
edb_pgbouncer.exe
edb_psqlodbc.exe
Explority Data Analysis.ipynb
Favorites
Links
Local Settings
Marketing Campaign Performance & Conversion Prediction.ipynb
Microsoft
Music
My Documents
NetHood
NTUSER.DAT
ntuser.dat.LOG1
ntuser.dat.LOG2
NTUSER.DAT{2ad838bc-efea-11ee-a54d-000d

In [36]:
from pathlib import Path

notebook_path = Path(
    "E:/Product_Analyst/notebooks/05_Automation_Pipeline.ipynb"
)

print("Notebook exists:", notebook_path.exists())

Notebook exists: False


In [37]:
from pathlib import Path

project = Path("E:/Product_Analyst")
notebooks_folder = project / "notebooks"

print("Project folder:", project.exists())
print("Notebooks folder:", notebooks_folder.exists())

Project folder: True
Notebooks folder: False


In [38]:
notebooks_folder.mkdir(parents=True, exist_ok=True)

print("Notebooks folder created!")

Notebooks folder created!


In [39]:
from pathlib import Path

notebook_path = Path(
    "E:/Product_Analyst/notebooks/05_Automation_Pipeline.ipynb"
)

print("Notebook exists:", notebook_path.exists())

Notebook exists: False


In [40]:
from pathlib import Path

for file in Path("C:/Users/kollu").rglob("Automation_Pipeline.ipynb"):
    print("Found:", file)

Found: C:\Users\kollu\Automation_Pipeline.ipynb


In [41]:
from pathlib import Path
import shutil

source = Path("C:/Users/kollu/Automation_Pipeline.ipynb")

destination = Path(
    "E:/Product_Analyst/notebooks/05_Automation_Pipeline.ipynb"
)

shutil.copy2(source, destination)

print("Notebook copied successfully!")
print("Destination exists:", destination.exists())
print("File size:", destination.stat().st_size, "bytes")

Notebook copied successfully!
Destination exists: True
File size: 46754 bytes


In [ ]:
import papermill as pm

output_path = (
    "E:/Product_Analyst/reports/05_Automation_Executed.ipynb"
)

pm.execute_notebook(
    str(destination),
    output_path,
    kernel_name="python3"
)

print("Notebook executed successfully!")

C:\Users\kollu\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Executing:  92%|█████████▏| 34/37 [00:20<00:00,  3.20cell/s]